# NewsLens AI · 01 · ISOT exploratory data analysis
This reproducible companion loads official CSVs from `data/raw/`. It never changes the original files. Run `python training/download_data.py --dataset isot` first.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
true = pd.read_csv(ROOT / 'data/raw/True.csv').assign(label='reliable')
fake = pd.read_csv(ROOT / 'data/raw/Fake.csv').assign(label='misleading')
raw = pd.concat([true, fake], ignore_index=True)
raw.shape, raw.columns.tolist()

In [ ]:
raw['combined'] = raw['title'].fillna('') + ' ' + raw['text'].fillna('')
raw['word_count'] = raw['combined'].str.split().str.len()
display(raw.groupby('label')['word_count'].describe().round(1))
print('Exact combined-text duplicates:', raw['combined'].str.lower().str.replace(r'\s+', ' ', regex=True).duplicated().sum())
print('Missing values by column:')
display(raw.isna().sum().to_frame('missing'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
raw['label'].value_counts().plot.bar(ax=axes[0], color=['#17C3E8', '#7657FF'], title='Raw class counts')
for label, group in raw.groupby('label'):
    group['word_count'].clip(upper=1500).plot.hist(ax=axes[1], bins=40, alpha=.55, label=label)
axes[1].set(title='Article length distribution (clipped at 1,500)', xlabel='Words')
axes[1].legend(); plt.tight_layout()

## Interpretation
ISOT contains strong source/topic asymmetry: reliable rows are largely wire-service copy, while fake rows come from different outlets and subjects. The production training script removes duplicates, excludes `subject`/source, neutralises Reuters/byline markers, uses a fixed stratified split, and fits TF-IDF only within each training fold. These controls reduce but cannot eliminate dataset shortcuts.